In [ ]:
import { useState, useEffect } from "react";

const RISK_PATTERNS = [
  {
    id: "urgency",
    label: "Urgency Manipulation",
    icon: "⚡",
    color: "#d63031",
    bgColor: "#fff5f5",
    borderColor: "#ffc9c9",
    patterns: [
      /urgent(ly)?/gi, /act now/gi, /immediately/gi, /asap/gi,
      /within \d+ (hours?|minutes?|days?)/gi, /time.sensitive/gi,
      /deadline/gi, /expire[sd]?/gi, /limited time/gi, /don't delay/gi,
      /respond (now|immediately|asap)/gi, /last chance/gi,
    ],
    description: "Creates false urgency to pressure quick action without thinking",
  },
  {
    id: "phishing",
    label: "Phishing Signals",
    icon: "🎣",
    color: "#e67e22",
    bgColor: "#fff8f0",
    borderColor: "#ffd8a8",
    patterns: [
      /verify your (account|identity|email|password|details)/gi,
      /confirm your (account|information|details|identity)/gi,
      /click (here|this link|below)/gi,
      /update your (account|payment|billing|information)/gi,
      /account (suspended|locked|disabled|compromised)/gi,
      /unusual (activity|sign.in|login)/gi,
      /security (alert|warning|breach)/gi,
      /your account (will be|has been)/gi,
    ],
    description: "Attempts to steal credentials or personal information",
  },
  {
    id: "sensitive_data",
    label: "Sensitive Data Request",
    icon: "🔐",
    color: "#6c5ce7",
    bgColor: "#f8f7ff",
    borderColor: "#d0c8ff",
    patterns: [
      /social security/gi, /ssn/gi, /credit card/gi, /card number/gi,
      /cvv/gi, /pin number/gi, /bank account/gi, /routing number/gi,
      /date of birth/gi, /mother.s maiden/gi, /password/gi,
      /full name and address/gi, /passport number/gi,
    ],
    description: "Requests highly sensitive personal or financial information",
  },
  {
    id: "suspicious_links",
    label: "Suspicious URLs",
    icon: "🔗",
    color: "#c0392b",
    bgColor: "#fff5f5",
    borderColor: "#ffb3b3",
    patterns: [
      /https?:\/\/[^
" :]{1,}\.\b(xyz|top|club|loan|work|click|link|gq|tk|ml|ga|cf)[^\s]*/gi,
      /https?:\/\/\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}/gi,
      /bit\.ly|tinyurl|t\.co|goo\.gl|ow\.ly/gi,
      /https?:\/\/[^
" :]*(login|signin|account|secure|update|verify)[^\s]*/gi,
    ],
    description: "Contains shortened, IP-based, or deceptive URLs",
  },
  {
    id: "threats",
    label: "Threats & Coercion",
    icon: "⚠️",
    color: "#b71c1c",
    bgColor: "#fff3f3",
    borderColor: "#ffcdd2",
    patterns: [
      /legal action/gi, /prosecut/gi, /arrest/gi, /lawsuit/gi,
      /police/gi, /irs/gi, /debt collector/gi, /overdue/gi,
      /penalty/gi, /fine of/gi, /suspend your/gi, /terminate your/gi,
      /consequences/gi, /court/gi,
    ],
    description: "Uses threats or intimidation to force compliance",
  },
  {
    id: "too_good",
    label: "Too-Good-To-Be-True",
    icon: "💰",
    color: "#2e7d32",
    bgColor: "#f1f8f1",
    borderColor: "#b9dfba",
    patterns: [
      /you('ve| have) (won|been selected|been chosen)/gi,
      /congratulations/gi, /lottery/gi, /prize/gi,
      /free (gift|money|iphone|vacation|cruise)/gi,
      /claim your/gi, /wire transfer/gi,
      /nigerian/gi, /inheritance/gi, /million dollar/gi,
      /make money (fast|quick|online)/gi, /work from home/gi,
    ],
    description: "Promises unrealistic rewards or opportunities",
  },
  {
    id: "impersonation",
    label: "Impersonation",
    icon: "🎭",
    color: "#1565c0",
    bgColor: "#f0f4ff",
    borderColor: "#bcd0f7",
    patterns: [
      /microsoft support/gi, /apple support/gi, /google security/gi,
      /amazon (customer|security|account)/gi, /paypal (security|team)/gi,
      /your (it|tech) (department|team)/gi, /hr department/gi,
      /ceo|cfo|executive team/gi, /official notice/gi,
    ],
    description: "Impersonates trusted organizations or authority figures",
  },
];

function analyzeEmail(text) {
  if (!text.trim()) return { score: 0, flags: [], riskLevel: "none", highlights: [] };
  const flags = [];
  const matchedRanges = [];
  for (const rule of RISK_PATTERNS) {
    const matches = [];
    for (const pattern of rule.patterns) {
      let match;
      const re = new RegExp(pattern.source, pattern.flags.includes("g") ? pattern.flags : pattern.flags + "g");
      while ((match = re.exec(text)) !== null) {
        matches.push({ start: match.index, end: match.index + match[0].length, text: match[0] });
        matchedRanges.push({ start: match.index, end: match.index + match[0].length, color: rule.color, id: rule.id });
      }
    }
    if (matches.length > 0) flags.push({ ...rule, count: matches.length, matches });
  }
  const score = Math.min(100, flags.reduce((acc, f) => acc + Math.min(f.count * 12, 30), 0));
  let riskLevel = "safe";
  if (score >= 70) riskLevel = "critical";
  else if (score >= 45) riskLevel = "high";
  else if (score >= 20) riskLevel = "medium";
  else if (score > 0) riskLevel = "low";
  return { score, flags, riskLevel, highlights: matchedRanges };
}

function highlightText(text, ranges) {
  if (!ranges.length) return [{ text, color: null }];
  const sorted = [...ranges].sort((a, b) => a.start - b.start);
  const parts = [];
  let cursor = 0;
  for (const r of sorted) {
    if (r.start > cursor) parts.push({ text: text.slice(cursor, r.start), color: null });
    parts.push({ text: text.slice(r.start, r.end), color: r.color });
    cursor = r.end;
  }
  if (cursor < text.length) parts.push({ text: text.slice(cursor), color: null });
  return parts;
}

const SAMPLE_EMAILS = [
  {
    label: "Phishing – Bank",
    content: `Subject: Urgent: Your Account Has Been Suspended\n\nDear Customer,\n\nWe have detected unusual activity on your bank account. Your account will be permanently suspended within 24 hours unless you verify your identity immediately.\n\nClick here to confirm your account: http://secure-bank-verify.xyz/login\n\nPlease provide your full name, date of birth, and bank account number to restore access.\n\nThis is an urgent security alert. Failure to respond will result in legal action.\n\nBank Security Team`,
  },
  {
    label: "Prize Scam",
    content: `CONGRATULATIONS! You have won our $1,000,000 lottery prize!\n\nYou've been selected from millions of entries as our grand prize winner. To claim your free gift, you must act now — this offer expires in 2 hours!\n\nClick this link: bit.ly/claim-prize-now\n\nWire transfer fees apply. Provide your credit card number and CVV to process. Contact our official team at prize@lottery-win.tk\n\nDon't delay — last chance to collect your winnings!`,
  },
  {
    label: "CEO Fraud",
    content: `From: CEO Office <ceo@company-official.top>\nSubject: Confidential Wire Transfer Request\n\nHi,\n\nThis is an urgent and confidential request from the CEO. We need to complete a wire transfer of $47,500 to a new vendor by end of day. Do not discuss this with anyone — legal action may follow if this is disclosed.\n\nPlease respond immediately with your bank account and routing number to process the transfer.\n\nExecutive Team`,
  },
  {
    label: "Legitimate Email",
    content: `Hi Sarah,\n\nFollowing up on our meeting from Tuesday — attached are the Q3 report slides we discussed.\n\nLet me know if you have any questions before the Thursday presentation. Happy to jump on a quick call if needed.\n\nBest,\nJames`,
  },
];

const RISK_CONFIG = {
  none:     { label: "No Content",  bg: "#f8f9fa", border: "#dee2e6", text: "#adb5bd", bar: "#dee2e6", headerBg: "#f1f3f5" },
  safe:     { label: "Safe",        bg: "#f0fdf4", border: "#86efac", text: "#16a34a", bar: "#22c55e", headerBg: "#dcfce7" },
  low:      { label: "Low Risk",    bg: "#fffbeb", border: "#fcd34d", text: "#d97706", bar: "#f59e0b", headerBg: "#fef3c7" },
  medium:   { label: "Medium Risk", bg: "#fff7ed", border: "#fdba74", text: "#ea580c", bar: "#f97316", headerBg: "#ffedd5" },
  high:     { label: "High Risk",   bg: "#fff1f2", border: "#fca5a5", text: "#dc2626", bar: "#ef4444", headerBg: "#fee2e2" },
  critical: { label: "CRITICAL",    bg: "#fff1f2", border: "#f87171", text: "#b91c1c", bar: "#dc2626", headerBg: "#fecaca" },
};

export default function App() {
  const [emailText, setEmailText] = useState("");
  const [result, setResult] = useState(null);
  const [analyzing, setAnalyzing] = useState(false);
  const [activeFlag, setActiveFlag] = useState(null);
  const [animScore, setAnimScore] = useState(0);

  const handleAnalyze = () => {
    if (!emailText.trim()) return;
    setAnalyzing(true);
    setResult(null);
    setAnimScore(0);
    setTimeout(() => {
      const r = analyzeEmail(emailText);
      setResult(r);
      setAnalyzing(false);
    }, 700);
  };

  useEffect(() => {
    if (!result) return;
    let frame;
    let cur = 0;
    const target = result.score;
    const step = () => {
      cur = Math.min(cur + 2, target);
      setAnimScore(cur);
      if (cur < target) frame = requestAnimationFrame(step);
    };
    frame = requestAnimationFrame(step);
    return () => cancelAnimationFrame(frame);
  }, [result]);

  const cfg = result ? RISK_CONFIG[result.riskLevel] : RISK_CONFIG.none;
  const highlightedParts = result && result.highlights.length > 0
    ? highlightText(emailText, result.highlights)
    : null;

  return (
    <div style={{
      minHeight: "100vh",
      background: "#f0f2f5",
      fontFamily: "'Segoe UI', system-ui, sans-serif",
      color: "#1a1a2e",
    }}>
      {/* Header */}
      <div style={{
        background: "#ffffff",
        borderBottom: "1px solid #e2e8f0",
        padding: "16px 32px",
        display: "flex",
        alignItems: "center",
        gap: "14px",
        boxShadow: "0 1px 4px rgba(0,0,0,0.06)",
      }}>
        <div style={{
          width: 42, height: 42,
          background: "linear-gradient(135deg, #e74c3c, #c0392b)",
          borderRadius: 10,
          display: "flex", alignItems: "center", justifyContent: "center",
          fontSize: 20,
          boxShadow: "0 2px 8px rgba(231,76,60,0.3)",
        }}>🛡️</div>
        <div>
          <div style={{ fontSize: 18, fontWeight: 700, color: "#1e293b", letterSpacing: 0.5 }}>
            Email Risk Analyzer
          </div>
          <div style={{ fontSize: 11, color: "#94a3b8", letterSpacing: 1, textTransform: "uppercase" }}>
            Threat Detection System
          </div>
        </div>
        {result && (
          <div style={{
            marginLeft: "auto",
            padding: "6px 16px",
            background: cfg.headerBg,
            border: `1.5px solid ${cfg.border}`,
            borderRadius: 20,
            fontSize: 12,
            fontWeight: 700,
            color: cfg.text,
            letterSpacing: 0.5,
          }}>
            {cfg.label}
          </div>
        )}
      </div>

      <div style={{ display: "grid", gridTemplateColumns: "1fr 380px", minHeight: "calc(100vh - 74px)" }}>
        {/* Left Panel */}
        <div style={{ padding: "24px 24px 24px 28px" }}>
          {/* Sample buttons */}
          <div style={{ marginBottom: 16 }}>
            <div style={{ fontSize: 11, fontWeight: 600, color: "#64748b", letterSpacing: 1, textTransform: "uppercase", marginBottom: 8 }}>
              Load Sample
            </div>
            <div style={{ display: "flex", gap: 8, flexWrap: "wrap" }}>
              {SAMPLE_EMAILS.map((s) => (
                <button key={s.label} onClick={() => { setEmailText(s.content); setResult(null); setAnimScore(0); }} style={{
                  background: "#ffffff",
                  border: "1.5px solid #e2e8f0",
                  color: "#475569",
                  padding: "6px 14px",
                  borderRadius: 6,
                  fontSize: 12,
                  cursor: "pointer",
                  fontFamily: "inherit",
                  fontWeight: 500,
                  transition: "all 0.15s",
                  boxShadow: "0 1px 2px rgba(0,0,0,0.04)",
                }}
                  onMouseEnter={e => { e.target.style.borderColor = "#e74c3c"; e.target.style.color = "#e74c3c"; e.target.style.background = "#fff5f5"; }}
                  onMouseLeave={e => { e.target.style.borderColor = "#e2e8f0"; e.target.style.color = "#475569"; e.target.style.background = "#ffffff"; }}
                >{s.label}</button>
              ))}
            </div>
          </div>

          {/* Textarea */}
          <div style={{ marginBottom: 14 }}>
            <div style={{ fontSize: 11, fontWeight: 600, color: "#64748b", letterSpacing: 1, textTransform: "uppercase", marginBottom: 8 }}>
              Email Content
            </div>
            <textarea
              value={emailText}
              onChange={e => { setEmailText(e.target.value); setResult(null); setAnimScore(0); }}
              placeholder={"Paste email content here...\n\nThe analyzer will detect:\n• Phishing attempts\n• Urgency manipulation\n• Sensitive data requests\n• Suspicious links\n• Threats & coercion\n• Impersonation tactics"}
              style={{
                width: "100%",
                height: 240,
                background: "#ffffff",
                border: "1.5px solid #e2e8f0",
                borderRadius: 8,
                color: "#334155",
                fontFamily: "'Courier New', monospace",
                fontSize: 13,
                padding: "14px 16px",
                resize: "vertical",
                outline: "none",
                lineHeight: 1.7,
                boxSizing: "border-box",
                boxShadow: "0 1px 3px rgba(0,0,0,0.04)",
                transition: "border-color 0.2s",
              }}
              onFocus={e => e.target.style.borderColor = "#94a3b8"}
              onBlur={e => e.target.style.borderColor = "#e2e8f0"}
            />
          </div>

          {/* Analyze button */}
          <button onClick={handleAnalyze} disabled={analyzing || !emailText.trim()} style={{
            background: emailText.trim()
              ? "linear-gradient(135deg, #e74c3c, #c0392b)"
              : "#e2e8f0",
            color: emailText.trim() ? "#ffffff" : "#94a3b8",
            border: "none",
            padding: "12px 28px",
            borderRadius: 8,
            fontSize: 13,
            fontWeight: 700,
            letterSpacing: 1,
            textTransform: "uppercase",
            cursor: emailText.trim() ? "pointer" : "not-allowed",
            fontFamily: "inherit",
            boxShadow: emailText.trim() ? "0 4px 12px rgba(231,76,60,0.3)" : "none",
            transition: "all 0.2s",
            width: "100%",
          }}>
            {analyzing ? "⟳  Scanning..." : "▶  Analyze Threat"}
          </button>

          {/* Threat Map */}
          {result && highlightedParts && (
            <div style={{ marginTop: 20 }}>
              <div style={{ fontSize: 11, fontWeight: 600, color: "#64748b", letterSpacing: 1, textTransform: "uppercase", marginBottom: 8 }}>
                Threat Map
              </div>
              <div style={{
                background: "#ffffff",
                border: "1.5px solid #e2e8f0",
                borderRadius: 8,
                padding: "14px 16px",
                fontSize: 12,
                lineHeight: 1.9,
                maxHeight: 210,
                overflowY: "auto",
                whiteSpace: "pre-wrap",
                wordBreak: "break-word",
                boxShadow: "0 1px 3px rgba(0,0,0,0.04)",
                color: "#475569",
              }}>
                {highlightedParts.map((p, i) => (
                  p.color
                    ? <mark key={i} style={{
                        background: p.color + "22",
                        color: p.color,
                        borderBottom: `2px solid ${p.color}`,
                        borderRadius: 3,
                        padding: "1px 3px",
                        fontWeight: 700,
                      }}>{p.text}</mark>
                    : <span key={i}>{p.text}</span>
                ))}
              </div>
            </div>
          )}
        </div>

        {/* Right Panel */}
        <div style={{ padding: "24px 20px", background: "#ffffff", borderLeft: "1px solid #e2e8f0" }}>
          {/* Risk Score Card */}
          <div style={{
            background: cfg.bg,
            border: `1.5px solid ${cfg.border}`,
            borderRadius: 10,
            padding: "20px",
            marginBottom: 18,
            transition: "all 0.4s",
          }}>
            <div style={{ display: "flex", justifyContent: "space-between", alignItems: "flex-start", marginBottom: 14 }}>
              <div>
                <div style={{ fontSize: 11, fontWeight: 600, color: "#94a3b8", letterSpacing: 1, textTransform: "uppercase", marginBottom: 4 }}>
                  Risk Score
                </div>
                <div style={{ fontSize: 44, fontWeight: 800, color: cfg.text, lineHeight: 1 }}>
                  {result ? animScore : "—"}
                  <span style={{ fontSize: 18, fontWeight: 400, color: "#94a3b8" }}> /100</span>
                </div>
              </div>
              {result && (
                <div style={{
                  padding: "5px 12px",
                  background: cfg.headerBg,
                  border: `1.5px solid ${cfg.border}`,
                  borderRadius: 20,
                  fontSize: 11,
                  color: cfg.text,
                  fontWeight: 700,
                  letterSpacing: 0.5,
                  marginTop: 4,
                }}>
                  {cfg.label}
                </div>
              )}
            </div>

            {/* Score bar */}
            <div style={{ background: "#e2e8f0", borderRadius: 4, height: 8, overflow: "hidden", marginBottom: 10 }}>
              <div style={{
                height: "100%",
                width: `${result ? animScore : 0}%`,
                background: `linear-gradient(90deg, ${cfg.bar}bb, ${cfg.bar})`,
                borderRadius: 4,
                transition: "width 0.05s",
              }} />
            </div>

            {result && (
              <div style={{ fontSize: 12, color: "#64748b", lineHeight: 1.5 }}>
                {result.riskLevel === "safe" && "✓ No suspicious patterns detected. Email appears legitimate."}
                {result.riskLevel === "low" && "△ Minor indicators present. Proceed with awareness."}
                {result.riskLevel === "medium" && "⚠ Multiple risk factors found. Verify before acting."}
                {result.riskLevel === "high" && "✗ High-risk content detected. Do not click links or share data."}
                {result.riskLevel === "critical" && "✗✗ CRITICAL: Multiple attack vectors detected. Delete immediately."}
              </div>
            )}
          </div>

          {/* Detected Threats */}
          <div style={{ fontSize: 11, fontWeight: 600, color: "#64748b", letterSpacing: 1, textTransform: "uppercase", marginBottom: 10 }}>
            Detected Threats {result ? `(${result.flags.length})` : ""}
          </div>

          {!result && (
            <div style={{ textAlign: "center", padding: "40px 0", color: "#cbd5e1" }}>
              {analyzing ? (
                <div>
                  <div style={{ fontSize: 28, marginBottom: 8 }}>⟳</div>
                  <div style={{ color: "#e74c3c", fontSize: 12, letterSpacing: 1, fontWeight: 600 }}>SCANNING...</div>
                </div>
              ) : (
                <div style={{ fontSize: 13 }}>
                  Paste email content<br />and run analysis
                </div>
              )}
            </div>
          )}

          {result && result.flags.length === 0 && (
            <div style={{
              background: "#f0fdf4",
              border: "1.5px solid #86efac",
              borderRadius: 8,
              padding: "16px",
              color: "#16a34a",
              fontSize: 13,
              textAlign: "center",
              fontWeight: 500,
            }}>
              ✓ No threat patterns detected
            </div>
          )}

          <div style={{ display: "flex", flexDirection: "column", gap: 8, maxHeight: "calc(100vh - 400px)", overflowY: "auto" }}>
            {result && result.flags.map(flag => (
              <div key={flag.id}
                onClick={() => setActiveFlag(activeFlag === flag.id ? null : flag.id)}
                style={{
                  background: activeFlag === flag.id ? flag.bgColor : "#f8fafc",
                  border: `1.5px solid ${activeFlag === flag.id ? flag.borderColor : "#e2e8f0"}`,
                  borderRadius: 8,
                  padding: "12px 14px",
                  cursor: "pointer",
                  transition: "all 0.15s",
                  boxShadow: activeFlag === flag.id ? `0 2px 8px ${flag.color}18` : "0 1px 2px rgba(0,0,0,0.03)",
                }}
                onMouseEnter={e => { e.currentTarget.style.borderColor = flag.borderColor; e.currentTarget.style.background = flag.bgColor; }}
                onMouseLeave={e => { if (activeFlag !== flag.id) { e.currentTarget.style.borderColor = "#e2e8f0"; e.currentTarget.style.background = "#f8fafc"; } }}
              >
                <div style={{ display: "flex", alignItems: "center", gap: 10 }}>
                  <span style={{ fontSize: 18 }}>{flag.icon}</span>
                  <div style={{ flex: 1 }}>
                    <div style={{ fontSize: 13, color: flag.color, fontWeight: 700 }}>
                      {flag.label}
                    </div>
                    <div style={{ fontSize: 11, color: "#94a3b8", marginTop: 1 }}>{flag.count} match{flag.count !== 1 ? "es" : ""}</div>
                  </div>
                  <div style={{
                    minWidth: 22, height: 22,
                    background: flag.bgColor,
                    border: `1.5px solid ${flag.borderColor}`,
                    borderRadius: "50%",
                    display: "flex", alignItems: "center", justifyContent: "center",
                    fontSize: 11, color: flag.color, fontWeight: 700,
                  }}>{flag.count}</div>
                </div>

                {activeFlag === flag.id && (
                  <div style={{ marginTop: 10, paddingTop: 10, borderTop: `1px solid ${flag.borderColor}` }}>
                    <div style={{ fontSize: 12, color: "#64748b", marginBottom: 8, lineHeight: 1.5 }}>
                      {flag.description}
                    </div>
                    <div style={{ display: "flex", flexWrap: "wrap", gap: 5 }}>
                      {flag.matches.slice(0, 6).map((m, i) => (
                        <span key={i} style={{
                          background: flag.bgColor,
                          color: flag.color,
                          border: `1px solid ${flag.borderColor}`,
                          borderRadius: 4,
                          padding: "2px 8px",
                          fontSize: 11,
                          fontFamily: "monospace",
                          fontWeight: 600,
                        }}>
                          "{m.text}"
                        </span>
                      ))}
                      {flag.matches.length > 6 && (
                        <span style={{ fontSize: 11, color: "#94a3b8", padding: "2px 4px" }}>
                          +{flag.matches.length - 6} more
                        </span>
                      )}
                    </div>
                  </div>
                )}
              </div>
            ))}
          </div>
        </div>
      </div>
    </div>
  );
}